In [ ]:
pip install unsloth transformers trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.6/314.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.9/233.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.5/132.5 kB 1

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

In [ ]:
model, tokenizer=FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True
)

==((====))==  Unsloth 2025.9.7: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:37: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
model=FastLanguageModel.get_peft_model(
    model,r=16,
    target_modules={"q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"}
)

Unsloth 2025.9.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
tokenizer=get_chat_template(tokenizer,chat_template="llama-3.1")

In [ ]:
dataset=load_dataset("mlabonne/FineTome-100k",split="train")

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [ ]:
dataset=standardize_sharegpt(dataset)

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/100000 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['conversations', 'source', 'score', 'text'],
    num_rows: 100000
})

In [ ]:
dataset[0]

{'conversations': [{'content': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.',
   'role': 'user'},
  {'content': 

In [ ]:
dataset = dataset.map(
    lambda examples: {
        "text": [
            tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
            for convo in examples["conversations"]
        ]
    },
    batched=True,
    num_proc=2
)

Map (num_proc=2):   0%|          | 0/100000 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['conversations', 'source', 'score', 'text'],
    num_rows: 100000
})

In [ ]:
dataset[0]

{'conversations': [{'content': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.',
   'role': 'user'},
  {'content': 

In [ ]:
trainer=SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        output_dir="outputs"
    ),
    packing=True,
    tokenizer=tokenizer
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: prahasith17 (prahasith17-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,1.439900
2,1.846100
3,1.380300
4,1.413900
5,1.350800
6,1.445800
7,0.946300
8,1.507200
9,1.239300
10,1.254000


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=60, training_loss=1.0265095988909403, metrics={'train_runtime': 604.934, 'train_samples_per_second': 0.793, 'train_steps_per_second': 0.099, 'total_flos': 5495762487078912.0, 'train_loss': 1.0265095988909403, 'epoch': 0.0048})

In [ ]:
model.save_pretrained("finetuned_model")

In [ ]:
inference_model, inference_tokenizer=FastLanguageModel.from_pretrained(
    model_name="./finetuned_model",
    max_seq_length=2048,
    load_in_4bit=True
)

==((====))==  Unsloth 2025.9.7: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:37: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


In [ ]:
text_prompts=[
    #"what are the key principles of investment?"
   # "what is the difference between recursion and dynamic programming"
    #"which waves travel faster sound or light ?"
   # "tell me a joke about programmers?"
    #"whats the point of life?"
    "give me an wrong information about java?"
]
formatted_prompts = []
for prompt in text_prompts:
    formatted_prompts.append(inference_tokenizer.apply_chat_template([
        {"role":"user","content":prompt}
    ],tokenize=False, add_generation_prompt=False
    ))
model_inputs=inference_tokenizer(formatted_prompts,return_tensors="pt").to("cuda")
generated_ids=inference_model.generate(
    **model_inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    top_p=0.95,
    pad_token_id=inference_tokenizer.eos_token_id
)
response=inference_tokenizer.batch_decode(generated_ids,skip_special_tokens=True)[0]
print(response)

system

Cutting Knowledge Date: December 2023
Today Date: 22 Sep 2025

user

give me an wrong information about java?assistant

Java is a programming language that was created in 1939 by a German physicist named Albert Einstein. Einstein was trying to create a language that could help him write mathematical formulas and equations more efficiently. He wanted a language that could handle complex calculations and was easy to read and write. Einstein developed the language and called it "Java," after a type of fruit that he loved. The language was initially called "Jabber" but was later renamed to Java. It was first released in 1939 and quickly became popular among scientists and engineers. Today, Java is one of the most widely used programming languages in the world.


**LLM Evaluation**


In [ ]:
!pip install evidently

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.5/236.5 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.8/510.8 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.2/452.2 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.8/69.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.6 MB/s eta 0:00:00


In [85]:
import pandas as pd
from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently.presets import TextEvals
from evidently.tests import lte, gte, eq
from evidently.descriptors import LLMEval, TestSummary, DeclineLLMEval, Sentiment, TextLength, IncludesWords
from evidently.llm.templates import BinaryClassificationPromptTemplate

In [ ]:
from evidently.ui.workspace import CloudWorkspace

In [ ]:
ws = CloudWorkspace(token="token here", url="https://app.evidently.cloud")

In [ ]:
project = ws.get_project("project id")

In [100]:
data = [
    ["what are the key principles of investment?", "The key principles of investment are: 1. **Diversification**: Investing in a variety of assets helps spread risk and potentially increase returns.2. **Risk management**: Understanding and managing risk is crucial to making informed investment decisions.3. **Long-term perspective**: Investing is a long-term game, and it's essential to avoid making impulsive decisions based on short-term market fluctuations.4. **Research and due diligence**: Investing in a company or asset requires thorough research and due diligence to ensure it aligns with your financial goals and risk tolerance.5. **Dollar-cost averaging**: Investing a fixed amount of money at regular intervals, regardless of market conditions, can help smooth out market volatility and reduce the impact of market downturns."],
    ["What is the capital of Japan?", "Key differences between recursion and dynamic programming:1.  Approach: Recursion involves solving a problem by breaking it down into smaller sub-problems and solving each one recursively. Dynamic programming involves storing the solutions to sub-problems in a table and using them to solve larger problems.2.  Storage: Recursion typically requires more memory, as each recursive call creates a new stack frame that stores the function"],
    ["Tell me a joke about programmers", "Why did the programmer quit his job? He didn't get arrays."],
    ["Whats the point of life?", "The question of the meaning of life is one that has been debated by philosophers, theologians, scientists, and many others for centuries. While there is no one definitive answer, there are several perspectives that attempt to provide insight into the purpose of life.1. Existentialism: This philosophical perspective suggests that life has no inherent meaning, and it is up to each individual to create their own purpose. Existentialists believe that people must take responsibility for their own lives and create their own meaning, rather than relying on external sources like religion or societal expectations.2. Utilitarianism: This ethical theory suggests that the meaning of life is to maximize overall happiness and well-being. From this perspective, life has a purpose insofar as it contributes to the greatest good for the greatest number of people.3. Nihilism: Nihilism is the belief that life has no inherent meaning or purpose. This perspective often arises from the rejection of traditional religious beliefs or the search for meaning in an inherently meaningless world.4. Religious perspectives: Many religious traditions offer their own answers to the meaning of life. For example, in Christianity, the meaning of life is often seen as serving God and following His commands. In Buddhism, the goal is to attain enlightenment and escape"],
    ["Give me an wrong information about java?", "Java is a programming language that was created in 1939 by a German physicist named Albert Einstein. Einstein was trying to create a language that could help him write mathematical formulas and equations more efficiently. He wanted a language that could handle complex calculations and was easy to read and write. Einstein developed the language and called it \"Java,\" after a type of fruit that he loved. The language was initially called Jabber but was later renamed to Java. It was first released in 1939 and quickly became popular among scientists and engineers. Today, Java is one of the most widely used programming languages in the world."]
]
columns = ["question", "response"]

eval_df = pd.DataFrame(data, columns=columns)
eval_df.head()

,question,response
0,what are the key principles of investment?,The key principles of investment are: 1. **Div...
1,What is the capital of Japan?,Key differences between recursion and dynamic ...
2,Tell me a joke about programmers,Why did the programmer quit his job? He didn't...
3,Whats the point of life?,The question of the meaning of life is one tha...
4,Give me an wrong information about java?,Java is a programming language that was create...


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "key"

In [101]:
from evidently.llm.templates import BinaryClassificationPromptTemplate, MulticlassClassificationPromptTemplate
from evidently.descriptors import LLMEval, ToxicityLLMEval, ContextQualityLLMEval, DeclineLLMEval, PIILLMEval,CorrectnessLLMEval

In [102]:
from evidently import DataDefinition
eval_df = Dataset.from_pandas(
  eval_df,
  data_definition=DataDefinition())

# Add a placeholder column for correct answers
eval_df.as_dataframe()["correct_answer"] = "" # You need to fill this with actual correct answers later

In [103]:
eval_df.add_descriptors(descriptors=[
    DeclineLLMEval("response", alias="refusal", include_reasoning=True),
    ToxicityLLMEval("response", alias="toxicity", include_category=False),
    PIILLMEval("response", alias="PII", include_score=True),
    CorrectnessLLMEval("response", target_output="correct_answer") # Add the target_output
])

In [104]:
eval_df.as_dataframe()

,question,response,correct_answer,refusal,refusal reasoning,toxicity reasoning,PII,PII score,PII reasoning,Correctness,Correctness reasoning
0,what are the key principles of investment?,The key principles of investment are: 1. **Div...,,OK,The text provides information about key invest...,The provided text discusses principles of inve...,OK,0.0,The text discusses general investment principl...,CORRECT,The OUTPUT accurately reflects the key princip...
1,What is the capital of Japan?,Key differences between recursion and dynamic ...,,OK,The text provides an explanation of the differ...,The provided text discusses the differences be...,OK,0.0,The text provided discusses technical concepts...,CORRECT,The provided text accurately conveys the disti...
2,Tell me a joke about programmers,Why did the programmer quit his job? He didn't...,,OK,The text appears to be a joke or a humorous st...,The text provided is a pun or a play on words ...,OK,0.0,The text does not contain any personally ident...,CORRECT,The OUTPUT accurately presents a joke about a ...
3,Whats the point of life?,The question of the meaning of life is one tha...,,OK,The text discusses various philosophical persp...,The text provided discusses philosophical pers...,OK,0.0,The text discusses philosophical perspectives ...,CORRECT,The OUTPUT accurately reflects the ideas prese...
4,Give me an wrong information about java?,Java is a programming language that was create...,,UNKNOWN,The provided text contains information about t...,The text provides a description of the Java pr...,OK,0.0,The text discusses the history and development...,INCORRECT,The OUTPUT contains factual inaccuracies. Java...
